In [1]:
import re
import random
import numpy as np
import pandas as pd
from langchain.text_splitter import RecursiveCharacterTextSplitter
from datasets import load_dataset
from datasets import concatenate_datasets
from sentence_transformers import SentenceTransformer

c:\Users\vallu\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
ds = load_dataset("rajpurkar/squad")
ds.column_names

{'train': ['id', 'title', 'context', 'question', 'answers'],
 'validation': ['id', 'title', 'context', 'question', 'answers']}

In [9]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
splitter = RecursiveCharacterTextSplitter(chunk_size=250,chunk_overlap=50)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3861.94it/s]


In [10]:
all_contexts = list(ds['train']['context']) + list(ds['validation']['context'])
all_query = list(ds['train']['question']) + list(ds['validation']['question'])

In [30]:
def add_label(example,label):
    example['label'] = label
    return example


def jaccard(q, c):
    q = set(q.lower().split())
    c = set(c.lower().split())
    return len(q & c) / len(q | c)


def query_coverage(q, c):
    q = set(q.lower().split())
    c = set(c.lower().split())
    return len(q & c) / len(q)


def shared_tokens(q, c):
    return len(
            set(q.lower().split()) &
            set(c.lower().split()))


def add_features(example):
    query = example.get('question', '')
    context = example.get('context', '')

    example['context'] = context #adding empty context

    example['jaccard']=jaccard(query,context)
    example['context_length']= len(context)
    example['query_coverage']=query_coverage(query,context)
    example['shared_tokens']=shared_tokens(query,context)

    return example

def swap_context(example):
    context = random.choice(all_contexts)
    while context == example["context"]:
        context = random.choice(all_contexts)

    example["context"] = context
    example["label"]=0
    
    return example


def cosine_similarity(vec1, vec2):
    vec1 = np.asarray(vec1)
    vec2 = np.asarray(vec2)

    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    if norm1 == 0 or norm2 == 0:
        return 0.0

    return np.dot(vec1, vec2)/(norm1 * norm2)

In [12]:
all_query_embeddings = model.encode(all_query)
all_chunks = []

for context in all_contexts:
    chunks = splitter.split_text(context)
    all_chunks.extend(chunks)

all_chunk_embeddings = model.encode(all_chunks)

query_cache = dict(zip(all_query,all_query_embeddings))
chunk_cache = dict(zip(all_chunks, all_chunk_embeddings))

In [13]:
def remove_best_chunk(example):
    context = example["context"]
    query = example["question"]

    chunks = splitter.split_text(context)
    if len(chunks) <= 1:
        return example

    query_emb = query_cache[query] if query in query_cache else model.encode(query)
    chunk_embeddings = [chunk_cache[chunk] if chunk in chunk_cache else model.encode(chunk) for chunk in chunks]


    similarities = [cosine_similarity(query_emb, ce) for ce in chunk_embeddings]
    best_idx = np.argmax(similarities)

    false_context = random.choice(all_contexts)
    while false_context == context:
        false_context = random.choice(all_contexts)
    
    false_chunks = splitter.split_text(false_context)

    if false_chunks:
        chunks[best_idx] = random.choice(false_chunks)

    example["context"] = " ".join(chunks)
    example["label"] = 0
    return example

In [14]:
ds = ds.remove_columns(["id", "title", "answers"])

In [15]:
ds["train"][0]

{'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?'}

In [16]:
ds = ds.map(lambda example: add_label(example, label=1))

In [17]:
negative_context_ds = ds.map(swap_context)
negative_context_ds_2 = ds.map(remove_best_chunk)

half = len(ds['train']) // 2

neg1 = negative_context_ds['train'].select(range(half))
neg2 = negative_context_ds_2['train'].select(range(len(ds['train']) - half))

half = len(ds['validation']) // 2

vneg1 = negative_context_ds['validation'].select(range(half))
vneg2 = negative_context_ds_2['validation'].select(range(len(ds['validation']) - half))

In [18]:
final_ds = concatenate_datasets([ds['train'], neg1, neg2,vneg1,vneg2]).shuffle(seed=42)

In [21]:
final_ds = final_ds.map(add_features)

Map: 100%|██████████| 185768/185768 [01:05<00:00, 2823.09 examples/s]


In [22]:
final_ds = final_ds.shuffle(seed=50)
final_ds

Dataset({
    features: ['context', 'question', 'label', 'jaccard', 'context_length', 'query_coverage', 'shared_tokens'],
    num_rows: 185768
})

In [23]:
df = final_ds.to_pandas()
df.to_csv('../datasets/SQuAD_processed.csv', index=False)

In [26]:
df.sample(5)

,context,question,label,jaccard,context_length,query_coverage,shared_tokens
115334,Season six premiered with the series' highest-...,Which television network originally aired The ...,0,0.024793,1127,0.375000,3
175282,The polarization of an antenna refers to the o...,When is a magnetic fields right angles to a el...,1,0.106061,677,0.700000,7
179203,Diagnosis of infectious disease sometimes invo...,How are minor infectious diseases treated?,1,0.045455,839,0.666667,4
142203,Muslim scientists placed far greater emphasis ...,What famous ship left Southampton's port carry...,0,0.010638,838,0.100000,1
137061,Buddhism /ˈbudɪzəm/ is a nontheistic religion[...,Who's teachings is Buddhism based upon?,1,0.043478,805,0.666667,4


This dataset doesn't cover questions which doesn't require context at all to tacle that I'm injecting a sample Natural QnA dataset.

In [59]:
ds = load_dataset("sentence-transformers/natural-questions")
ds

DatasetDict({
    train: Dataset({
        features: ['query', 'answer'],
        num_rows: 100231
    })
})

In [60]:
ds = ds['train']
ds = ds.remove_columns(['answer'])
ds = ds.rename_column('query', 'question')

In [61]:
ds = ds.map(lambda example: add_label(example, label=1))
ds = ds.map(add_features)

In [62]:
ds = ds.shuffle(seed=50)
ds[0]

{'question': 'how to say praise be to allah in arabic',
 'label': 1,
 'context': '',
 'jaccard': 0.0,
 'context_length': 0,
 'query_coverage': 0.0,
 'shared_tokens': 0}

In [70]:
qna_df = ds.to_pandas()
main_df = pd.read_csv('../datasets/SQuAD_processed.csv')

qna_sample = qna_df.sample(n=30000, random_state=42)

with this I also need to inject some samples with need context but have zero context if I won't the model will learn that when ever you see context = "" mark it as label 1.

In [71]:
needs_context_no_context = main_df[main_df['label'] == 1].sample(n=10000, random_state=42).copy()
needs_context_no_context['context'] = ''
needs_context_no_context['label'] = 0
needs_context_no_context['jaccard'] = 0.0
needs_context_no_context['context_length'] = 0
needs_context_no_context['query_coverage'] = 0.0
needs_context_no_context['shared_tokens'] = 0

In [72]:
merged_df = pd.concat([main_df, qna_sample, needs_context_no_context], ignore_index=True)

In [73]:
merged_df['context'] = merged_df['context'].fillna('')

In [76]:
merged_df.sample(10)

,context,question,label,jaccard,context_length,query_coverage,shared_tokens
180509,"signature dome and chimneys, represented a ste...","In 1922, which estate sold The Times?",0,0.039216,369,0.285714,2
7695,The city's population increased more than sixf...,Studies published in 2007 and 2008 support wha...,0,0.081633,915,0.615385,8
204293,,the most double doubles in an nba season,1,0.000000,0,0.000000,0
37276,"Beginning in April 1985, Madonna embarked on h...",When was the Live Aid Charity Concert held?,1,0.052174,889,0.750000,6
14796,Regular script typefaces are also commonly use...,What resembles an actual person's handwriting?,1,0.015152,545,0.166667,1
168257,The Theater District is a 17-block area in the...,What tourist offerings does the Space Center h...,0,0.043011,902,0.500000,4
158212,The official language of Bern is (the Swiss va...,What dialect is Bernese German?,1,0.136364,162,0.600000,3
11413,"For exercise, Tesla walked between 8 to 10 mil...",Where does the brain usually get most of its e...,0,0.025000,166,0.076923,1
18957,Holy Cross Father John Francis O'Hara was elec...,For whos glory did Father O'Hara believed that...,1,0.111111,744,0.785714,11
137563,155th Street starts on the West Side at Rivers...,In what year was the viaduct along 155th Stree...,1,0.058824,551,0.400000,4


In [77]:
merged_df.to_csv('../datasets/SQuAD_processed.csv', index=False)

Adding SQuAD 2.0, for some more unanswerable questions

In [101]:
squad_ds = load_dataset("rajpurkar/squad_v2")

In [102]:
squad_ds

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})

In [103]:
squad_ds['train'][0]

{'id': '56be85543aeaaa14008c9063',
 'title': 'Beyoncé',
 'context': 'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".',
 'question': 'When did Beyonce start becoming popular?',
 'answers': {'text': ['in the late 1990s'], 'answer_start': [269]}}

In [104]:
squad2_unanswerable = squad_ds.filter(lambda x: len(x['answers']['text']) == 0)
squad2_unanswerable

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 43498
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 5945
    })
})

In [105]:
squad2_unanswerable = concatenate_datasets([squad2_unanswerable['train'], squad2_unanswerable['validation']])
squad2_unanswerable = squad2_unanswerable.remove_columns(['id', 'title', 'answers'])

In [106]:
squad2_unanswerable = squad2_unanswerable.map(lambda example: add_label(example, label=0))
squad2_unanswerable = squad2_unanswerable.map(add_features)

In [107]:
squad2_unanswerable

Dataset({
    features: ['context', 'question', 'label', 'jaccard', 'context_length', 'query_coverage', 'shared_tokens'],
    num_rows: 49443
})

In [108]:
squad2_unanswerable[0]

{'context': 'The Legend of Zelda: Twilight Princess (Japanese: ゼルダの伝説 トワイライトプリンセス, Hepburn: Zeruda no Densetsu: Towairaito Purinsesu?) is an action-adventure game developed and published by Nintendo for the GameCube and Wii home video game consoles. It is the thirteenth installment in the The Legend of Zelda series. Originally planned for release on the GameCube in November 2005, Twilight Princess was delayed by Nintendo to allow its developers to refine the game, add more content, and port it to the Wii. The Wii version was released alongside the console in North America in November 2006, and in Japan, Europe, and Australia the following month. The GameCube version was released worldwide in December 2006.[b]',
 'question': 'What category of game is Legend of Zelda: Australia Twilight?',
 'label': 0,
 'jaccard': 0.0821917808219178,
 'context_length': 705,
 'query_coverage': 0.6666666666666666,
 'shared_tokens': 6}

In [ ]:
main_df = pd.read_csv('../datasets/SQuAD_processed.csv')
squad2_unanswerable = squad2_unanswerable.to_pandas()

merged_df = pd.concat([main_df, squad2_unanswerable], ignore_index=True)

In [112]:
merged_df['context'] = merged_df['context'].fillna('')
merged_df.sample(10)

,context,question,label,jaccard,context_length,query_coverage,shared_tokens
200768,,can i use my myki card in sydney,1,0.000000,0,0.000000,0
24640,Jehovah's Witnesses have been accused of havin...,What are Jehovah's Witnesses accused of concea...,1,0.043478,1072,0.500000,5
115015,"On November 21, 1789, North Carolina became th...",North Carolina was the twelth state to ratify ...,1,0.061404,1118,0.700000,7
84457,Chopin's life was covered in a BBC TV document...,What two people created a documentary on Chopi...,1,0.151515,193,0.454545,5
81538,The earliest examples of neoclassical architec...,In what town is the oldest neoclassical archit...,1,0.085714,655,0.666667,6
109501,"The period preceding, and contemporary with, t...",Which color of hair is most common among mamma...,0,0.010204,896,0.100000,1
214675,,what does being born with an extra y chromosom...,1,0.000000,0,0.000000,0
41560,categories in which to further its efforts to ...,When trees have a visible difference in color ...,0,0.123077,577,0.533333,8
163520,Oklahoma City is home to the state's largest s...,What is Oklahoma's largest school district?,1,0.032609,862,0.500000,3
128056,Bond disobeys M's order and travels to Rome to...,Where does Bond go after his suspension?,1,0.027027,594,0.285714,2


In [113]:
print(merged_df['label'].value_counts())
print(merged_df['label'].value_counts(normalize=True))

label
1    169819
0    155392
Name: count, dtype: int64
label
1    0.522181
0    0.477819
Name: proportion, dtype: float64


In [114]:
merged_df.to_csv('../datasets/SQuAD_processed.csv', index=False)